In [ ]:
import os
import numpy as np

#Open libsgma, onyxia
import sys
sys.path.append('/home/onyxia/work')
from libsigma import classification as cla
from libsigma import read_and_write as rw
from libsigma import plots


from osgeo import gdal
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.tree import DecisionTreeClassifier # ou RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import geopandas as gpd #ler arquivos vetoriais

In [8]:
# 1. Defina a lista de caminhos de arquivo
# **Substitua este caminho base pelo caminho real onde seus arquivos estão localizados**
base_dir = '/home/onyxia/work/data/projet_eval/'

# Lista dos arquivos que você deseja abrir (exemplo)
filenames = [
    'pyrenees_23-24_B02.tif',
    'pyrenees_23-24_B03.tif',
    'pyrenees_23-24_B04.tif',
    'pyrenees_23-24_B05.tif',
    'pyrenees_23-24_B06.tif',
    'pyrenees_23-24_B07.tif',
    'pyrenees_23-24_B08.tif',
    'pyrenees_23-24_B8A.tif',
    'pyrenees_23-24_B11.tif',
    'pyrenees_23-24_B12.tif'
    ]

# Crie um dicionário para armazenar os caminhos e os objetos GDAL abertos
# 'caminho_do_arquivo': objeto_gdal
gdal_datasets = {}

# 2. Itere sobre a lista usando um loop 'for'
for filename in filenames:
    # 3. Construa o caminho completo do arquivo
    full_path = os.path.join(base_dir, filename)

    # 4. Abra o arquivo
    try:
        # Usando gdal.GA_ReadOnly se você só for ler, ou GA_Update se for modificar
        data_set = gdal.Open(full_path, gdal.GA_Update)

        if data_set is not None:
            print(f"✅ Arquivo aberto com sucesso: {full_path}")
            # 5. Salve o caminho completo e o objeto data_set no dicionário
            gdal_datasets[full_path] = data_set
        else:
            print(f"❌ Não foi possível abrir o arquivo: {full_path}")

    except Exception as e:
        print(f"🛑 Erro ao tentar abrir {full_path}: {e}")

# Exemplo de como acessar um dos datasets abertos:
# (Isto é útil se você precisar processar os dados mais tarde)
if gdal_datasets:
    # Obtém o caminho da primeira chave (primeiro arquivo na lista)
    primeiro_caminho = list(gdal_datasets.keys())[0]
    primeiro_dataset = gdal_datasets[primeiro_caminho]

    print("\n--- Informações do primeiro arquivo aberto ---")
    print(f"Caminho: {primeiro_caminho}")
    print(f"Número de bandas: {primeiro_dataset.RasterCount}")
    print(f"Driver: {primeiro_dataset.GetDriver().LongName}")

# **IMPORTANTE: Feche os datasets quando terminar de usá-los!**
# Isso é crucial para liberar recursos e garantir que as alterações sejam salvas (se GA_Update foi usado).
for path, dataset in gdal_datasets.items():
    dataset = None # Fechar a referência ao dataset
    print(f"Fechado: {path}")



✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B02.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B03.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B04.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B05.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B06.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B07.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B08.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B8A.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B11.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B12.tif

--- Informações do primeiro arquivo aberto ---
Caminho: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B02.tif
Númer

In [12]:
len(gdal_datasets)

10

In [16]:
base_data = "/home/onyxia/work/data"
results_dir = "results"
fig_dir = os.path.join(results_dir, "figure")

os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)


samples_file = os.path.join(base_data, "PI_strates_pyrenees_32630.shp")


In [23]:
from osgeo import gdal
import os
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display # Mantenha se estiver usando Jupyter/IPython

# --- DEFINIÇÕES DE FUNÇÕES FALTANTES (Substitua pela sua lógica real) ---
# Se você tiver essas funções em outro arquivo, importe-as.
# Caso contrário, defina-as aqui (mesmo que sejam placeholders temporários).
def count_polygons_by_class(gdf):
    """Conta o número de feições (polígonos) por classe no GeoDataFrame."""
    # Sua lógica real aqui. Exemplo:
    # return gdf['CLASSE'].value_counts()
    return gdf['CLASSE'].value_counts()

def rasterize_samples(gdf):
    """Simula a rasterização e contagem de pixels. Substitua pela sua lógica GDAL/Rasterização."""
    # Sua lógica real aqui. Exemplo:
    # return (gdf['CLASSE'].value_counts() * 1000) # Exemplo simples
    counts = gdf['CLASSE'].value_counts()
    return counts * 100 # Simulação para pixels
# --- FIM DAS DEFINIÇÕES DE FUNÇÕES FALTANTES ---

def processar_e_visualizar_dados_vetoriais(base_data, results_dir, samples_file_name):
    """
    Função principal para carregar dados vetoriais (GeoJSON/Shapefile),
    processar contagens e gerar visualizações.
    """
    print("\n--- INICIANDO PROCESSAMENTO VETORIAL ---")

    # 1. Configurar diretórios de saída
    fig_dir = os.path.join(results_dir, "figure")
    os.makedirs(results_dir, exist_ok=True)
    os.makedirs(fig_dir, exist_ok=True)
    print(f"Diretórios criados/verificados: {results_dir} e {fig_dir}")

    # 2. Carregar o arquivo GeoJSON/Shapefile
    samples_file = os.path.join(base_data, samples_file_name)
    print(f"Tentando carregar o arquivo: {samples_file}")

    try:
        gdf = gpd.read_file(samples_file)
        print(f"✅ GeoDataFrame carregado com {len(gdf)} feições.")
    except Exception as e:
        print(f"❌ Erro ao carregar o arquivo vetorial: {e}")
        return

    # 3. Processamento e Visualização de Contagem de Polígonos
    poly_counts = count_polygons_by_class(gdf)
    plot_bar_chart(poly_counts, fig_dir, "poly", "polygones")
    display(poly_counts)

    # 4. Processamento e Visualização de Contagem de Pixels (Rasterização)
    pix_counts = rasterize_samples(gdf)
    plot_bar_chart(pix_counts, fig_dir, "pix", "pixels")
    display(pix_counts)

    print("--- PROCESSAMENTO VETORIAL CONCLUÍDO ---")


def plot_bar_chart(counts_series, fig_dir, prefix, label_y):
    """
    Função auxiliar para gerar e salvar um gráfico de barras.
    """
    
    file_name = f"diag_baton_nb_{prefix}_by_class.png"
    
    plt.figure(figsize=(8,5))
    plt.bar(counts_series.index, counts_series.values)
    plt.xlabel("id")
    plt.ylabel(f"Nombre de {label_y}")
    plt.title(f"Nombre de {label_y} par classe")
    
    save_path = os.path.join(fig_dir, file_name)
    plt.savefig(save_path)
    print(f"Gráfico salvo em: {save_path}")
    
    # plt.show() # Descomente se quiser exibir o gráfico imediatamente
    plt.close() # Fecha a figura para liberar memória


# --- FUNÇÃO GDAL ORIGINAL REORGANIZADA PARA MELHOR PADRONIZAÇÃO ---

def abrir_datasets_gdal(base_dir, filenames):
    """
    Itera sobre uma lista de arquivos e os abre usando gdal.Open.
    Retorna um dicionário com os datasets abertos.
    """
    print("\n--- INICIANDO ABERTURA DE DATASETS GDAL ---")
    gdal_datasets = {}

    for filename in filenames:
        full_path = os.path.join(base_dir, filename)

        try:
            # gdal.GA_Update para abrir em modo de leitura e escrita
            data_set = gdal.Open(full_path, gdal.GA_Update)

            if data_set is not None:
                print(f"✅ Arquivo aberto com sucesso: {full_path}")
                gdal_datasets[full_path] = data_set
            else:
                # O arquivo existe, mas o GDAL não conseguiu ler (formato errado, etc.)
                print(f"❌ Não foi possível abrir o arquivo (dataset vazio): {full_path}")

        except Exception as e:
            # Erro de I/O, permissão, etc.
            print(f"🛑 Erro ao tentar abrir {full_path}: {e}")

    print("--- ABERTURA DE DATASETS GDAL CONCLUÍDA ---")
    return gdal_datasets


def fechar_datasets_gdal(gdal_datasets):
    """
    Fecha todos os datasets GDAL abertos no dicionário.
    """
    print("\n--- FECHANDO DATASETS GDAL ---")
    for path, dataset in gdal_datasets.items():
        dataset = None # Fechar a referência ao dataset
        print(f"Fechado: {path}")
    print("--- DATASETS GDAL FECHADOS ---")


# --- BLOCO PRINCIPAL DE EXECUÇÃO ---

if __name__ == "__main__":
    
    # --- PARTE 1: Configurações e Execução GDAL (Raster) ---
    base_dir_raster = '/home/onyxia/work/data/projet_eval'
    filenames_raster = [
        'pyrenees_23-24_B02.tif',
    'pyrenees_23-24_B03.tif',
    'pyrenees_23-24_B04.tif',
    'pyrenees_23-24_B05.tif',
    'pyrenees_23-24_B06.tif',
    'pyrenees_23-24_B07.tif',
    'pyrenees_23-24_B08.tif',
    'pyrenees_23-24_B8A.tif',
    'pyrenees_23-24_B11.tif',
    'pyrenees_23-24_B12.tif' 
      ]

    datasets_abertos = abrir_datasets_gdal(base_dir_raster, filenames_raster)
    
    # Processamento dos datasets abertos ocorreria aqui...
    # Exemplo: for path, ds in datasets_abertos.items(): ...

    # Fechar os datasets ao final
    fechar_datasets_gdal(datasets_abertos)


    # --- PARTE 2: Configurações e Execução GeoPandas/Matplotlib (Vetorial) ---
    
    # Mantendo as variáveis originais para a segunda parte
    base_data_vector = "/home/onyxia/work/data/projet_eval"
    results_dir_vector = "results"
    samples_file_name_vector = "PI_strates_pyrenees_32630.shp"
    
    processar_e_visualizar_dados_vetoriais(
        base_data_vector,
        results_dir_vector,
        samples_file_name_vector
    )


--- INICIANDO ABERTURA DE DATASETS GDAL ---
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B02.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B03.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B04.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B05.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B06.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B07.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B08.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B8A.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B11.tif
✅ Arquivo aberto com sucesso: /home/onyxia/work/data/projet_eval/pyrenees_23-24_B12.tif
--- ABERTURA DE DATASETS GDAL CONCLUÍDA ---

--- FECHANDO DATASETS GDAL ---

KeyError: 'CLASSE'

In [21]:
import os
import geopandas as gpd

def listar_colunas_do_shapefile(base_data, samples_file_name):
    """
    Carrega o arquivo shapefile e imprime todos os nomes das colunas
    (incluindo a coluna 'geometry').
    """
    print("\n--- INICIANDO VERIFICAÇÃO DE COLUNAS VETORIAIS ---")

    # 1. Constrói o caminho completo do arquivo
    samples_file = os.path.join(base_data, samples_file_name)
    print(f"Tentando carregar o arquivo: {samples_file}")

    # 2. Carrega o arquivo usando GeoPandas
    try:
        # Tenta carregar o arquivo
        gdf = gpd.read_file(samples_file)
        print(f"✅ GeoDataFrame carregado com sucesso ({len(gdf)} feições).")

    except Exception as e:
        print(f"❌ Erro ao carregar o arquivo vetorial: {e}")
        # Retorna uma lista vazia em caso de falha
        return []

    # 3. Extrai e exibe os nomes das colunas
    colunas = gdf.columns.tolist()
    
    print("\n==============================================")
    print("📢 NOMES DAS COLUNAS DISPONÍVEIS NO SHAPEFILE:")
    print("==============================================")
    for coluna in colunas:
        print(f" - {coluna}")
    print("==============================================")
    
    return colunas

# --- BLOCO PRINCIPAL DE EXECUÇÃO ---

if __name__ == "__main__":
    
    # --- PARTE VETORIAL: Configurações ---
    
    # 🚨 Use o caminho exato onde o seu arquivo está.
    base_data_vector = "/home/onyxia/work/data/projet_eval"
    samples_file_name_vector = "PI_strates_pyrenees_32630.shp"
    
    # Executa a função
    colunas_encontradas = listar_colunas_do_shapefile(
        base_data_vector,
        samples_file_name_vector
    )
    
    # Exemplo de como usar a informação:
    if colunas_encontradas:
        print(f"\nTotal de colunas encontradas: {len(colunas_encontradas)}")
        
        # Se você precisar encontrar o nome da sua coluna de classe, por exemplo:
        # nome_da_classe = colunas_encontradas[1] 
        # print(f"Segunda coluna (exemplo de classe): {nome_da_classe}")


--- INICIANDO VERIFICAÇÃO DE COLUNAS VETORIAIS ---
Tentando carregar o arquivo: /home/onyxia/work/data/projet_eval/PI_strates_pyrenees_32630.shp
✅ GeoDataFrame carregado com sucesso (206 feições).

📢 NOMES DAS COLUNAS DISPONÍVEIS NO SHAPEFILE:
 - id
 - strate
 - comment
 - geometry

Total de colunas encontradas: 4
